## The highway layer

A plain layer computes $y = H(x, W_H)$. A highway layer adds two gates:

$$
y = H(x, W_H) \odot T(x, W_T) + x \odot C(x, W_C),
\qquad C(x, W_C) = 1 - T(x, W_T).
$$

The paper uses a sigmoid transform gate,

$$
T(x) = \sigma(W_T^\top x + b_T),
$$

so every unit has its own gate value between 0 and 1. The layer therefore interpolates between two
behaviours:

$$
y =
\begin{cases}
x, & \text{if } T(x) = 0 \quad \text{(carry the input)},\\
H(x, W_H), & \text{if } T(x) = 1 \quad \text{(transform the input)}.
\end{cases}
$$

The same is true for the Jacobian: when $T(x)=0$ the layer's Jacobian is the identity, so gradients
can travel through it unchanged.

For Equation (3) to work, $x$, $y$, $H(x)$ and $T(x)$ must have the same shape. This is why the
paper starts with a plain layer that changes the dimensionality and then stacks highway layers.

**The important initialization detail.** The bias $b_T$ starts at a negative value (the paper
uses values such as $-1$ and $-3$; we use $-2$). Since $\sigma(-2) \approx 0.12$, the gates start
mostly closed and the deep network initially behaves like a shallow one. The gates then open up
during training where they are useful.

## Setup

In [30]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

torch 2.15.0+cu132 | CUDA available: True
NVIDIA GeForce RTX 3060 Ti


In [31]:
SEED = 0
DATA_ROOT = Path("data")
BATCH_SIZE = 128
EPOCHS = 150
LR = 0.1
MOMENTUM = 0.9
LR_DECAY = 0.98
CLIP_NORM = 10.0
DEPTH = 12
HIGHWAY_WIDTH = 512
PLAIN_WIDTH = 692  # wider, so both models have about the same number of parameters
GATE_BIAS = -2.0
NUM_WORKERS = 2  # set to 0 on Windows if the data loader hangs


def set_seed(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Data

CIFAR-10 has 50,000 training and 10,000 test images of size $32 \times 32$, in 10 classes.
Training images get random crops and horizontal flips, which approximate the paper's random
translations and mirroring. Both splits are normalized with the dataset mean and standard
deviation.

In [32]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]
)
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_set = datasets.CIFAR10(DATA_ROOT, train=True, download=True, transform=train_tf)
test_set = datasets.CIFAR10(DATA_ROOT, train=False, download=True, transform=test_tf)
classes = test_set.classes

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)
test_loader = DataLoader(
    test_set,
    batch_size=512,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=device.type == "cuda",
)

# A fixed batch used later to inspect the gates.
probe_x, _ = next(iter(test_loader))
probe_x = probe_x[:256].to(device)

print(f"{len(train_set):,} train / {len(test_set):,} test images")

50,000 train / 10,000 test images


## The highway layer in code

The code is a direct translation of Equation (3): one linear layer produces $H$, one produces the
gate logits, and the sigmoid turns those into $T$. The full gate tensor is stored so we can inspect
it after training.

In [33]:
class Highway(nn.Module):
    def __init__(self, dim: int, gate_bias: float = GATE_BIAS):
        super().__init__()
        self.H = nn.Linear(dim, dim)
        self.gate = nn.Linear(dim, dim)
        nn.init.constant_(self.gate.bias, gate_bias)  # start biased towards carry
        self.last_gate = torch.zeros(dim)  # filled on every forward pass
        self.gate_off = False  # used later to switch this layer off

    def forward(self, x):
        h = F.relu(self.H(x))
        t = torch.sigmoid(self.gate(x))
        if self.gate_off:
            t = torch.zeros_like(t)
        self.last_gate = t.detach()
        return h * t + x * (1 - t)

## Plain and highway networks

Both models have the paper's overall shape:

* a plain first layer that changes the dimensionality (`Flatten -> Linear -> ReLU`),
* `DEPTH` hidden blocks,
* a linear classifier.

Only the hidden blocks differ. A highway block has two linear maps, so the plain model is made
wider (692 instead of 512 units) to keep the parameter counts comparable. The paper does the same
with widths 71 and 50; this notebook uses the same ratio at a larger scale.

To keep the notebook small, both models are fully connected. The paper's CIFAR-10 models are
convolutional, but the highway mechanism and the training setup are the same.

All linear weights use He initialization, as in the paper. The transform-gate biases are reset to
`-2` after initialization, so the gates start mostly closed.

In [34]:
class Net(nn.Module):
    def __init__(self, highway: bool, width: int, depth: int = DEPTH):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Flatten(),
            nn.Linear(3 * 32 * 32, width),
            nn.ReLU(),
        )
        if highway:
            self.blocks = nn.ModuleList(Highway(width) for _ in range(depth))
        else:
            self.blocks = nn.ModuleList(
                nn.Sequential(nn.Linear(width, width), nn.ReLU()) for _ in range(depth)
            )
        self.head = nn.Linear(width, 10)

    def forward(self, x):
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x)


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def init_weights(model):
    """He initialization for every linear layer, then reset the gate biases."""
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
            nn.init.zeros_(module.bias)
    for module in model.modules():
        if isinstance(module, Highway):
            nn.init.constant_(module.gate.bias, GATE_BIAS)

## Training

The setup follows the paper:

* SGD with momentum 0.9,
* batch size 128,
* He initialization for all linear weights,
* exponential learning-rate decay by a factor of 0.98 per epoch,
* 150 epochs and no weight decay.

The paper selected its learning rate, momentum, and decay with a random search. The values here
are a representative setup. We also clip the gradient norm at 10. The paper does not mention
clipping, but it keeps SGD stable in this fully connected setting.

For the highway model we also record the mean gate value of every layer after each epoch.

In [35]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(dim=1) == y).sum().item()
        total += y.numel()
    return correct / total


@torch.no_grad()
def gate_means_on_batch(model, x):
    model.eval()
    model(x)
    return [block.last_gate.mean().item() for block in model.blocks]


def train(model, track_gates: bool = False, epochs: int = EPOCHS, lr: float = LR):
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=MOMENTUM)
    sched = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=LR_DECAY)
    loss_fn = nn.CrossEntropyLoss()

    history = {"loss": [], "acc": [], "gates": []}
    start = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(x), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            opt.step()
            running += loss.item() * y.numel()
        history["loss"].append(running / len(train_set))
        history["acc"].append(accuracy(model, test_loader))
        if track_gates:
            history["gates"].append(gate_means_on_batch(model, probe_x))
        print(
            f"epoch {epoch:3d}/{epochs} | lr {opt.param_groups[0]['lr']:.4f} "
            f"| loss {history['loss'][-1]:.3f} | test acc {history['acc'][-1]:.1%}"
        )
        sched.step()

    print(f"trained in {time.time() - start:.1f}s")
    return history

## Train the baseline

In [36]:
set_seed()
plain = Net(highway=False, width=PLAIN_WIDTH)
init_weights(plain)
print(f"plain:   {n_params(plain):,} parameters")
plain_hist = train(plain)

plain:   7,888,118 parameters
epoch   1/150 | lr 0.1000 | loss 2.095 | test acc 26.3%
epoch   2/150 | lr 0.0980 | loss 1.910 | test acc 27.1%
epoch   3/150 | lr 0.0960 | loss 1.856 | test acc 32.9%
epoch   4/150 | lr 0.0941 | loss 1.815 | test acc 36.4%
epoch   5/150 | lr 0.0922 | loss 1.784 | test acc 36.1%
epoch   6/150 | lr 0.0904 | loss 1.746 | test acc 38.7%
epoch   7/150 | lr 0.0886 | loss 1.709 | test acc 40.0%
epoch   8/150 | lr 0.0868 | loss 1.681 | test acc 40.4%
epoch   9/150 | lr 0.0851 | loss 1.669 | test acc 40.6%
epoch  10/150 | lr 0.0834 | loss 1.639 | test acc 41.8%
epoch  11/150 | lr 0.0817 | loss 1.633 | test acc 43.2%
epoch  12/150 | lr 0.0801 | loss 1.611 | test acc 41.9%
epoch  13/150 | lr 0.0785 | loss 1.600 | test acc 44.9%
epoch  14/150 | lr 0.0769 | loss 1.575 | test acc 44.5%
epoch  15/150 | lr 0.0754 | loss 1.561 | test acc 44.3%
epoch  16/150 | lr 0.0739 | loss 1.548 | test acc 44.9%
epoch  17/150 | lr 0.0724 | loss 1.534 | test acc 44.2%
epoch  18/150 | lr

KeyboardInterrupt: 

## Train the highway network

In [ ]:
set_seed()
highway = Net(highway=True, width=HIGHWAY_WIDTH)
init_weights(highway)
print(f"highway: {n_params(highway):,} parameters")
highway_hist = train(highway, track_gates=True)

## Results

Two identical training setups, with and without the gates.

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(10, 3.5))

for name, hist in [("plain", plain_hist), ("highway", highway_hist)]:
    ax_loss.plot(hist["loss"], label=name)
    ax_acc.plot([100 * a for a in hist["acc"]], label=name)

ax_loss.set(xlabel="epoch", ylabel="train loss", title="Training loss")
ax_acc.set(xlabel="epoch", ylabel="test accuracy (%)", title="Test accuracy")
for ax in (ax_loss, ax_acc):
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

for name, hist in [("plain", plain_hist), ("highway", highway_hist)]:
    print(f"{name:8s} final test accuracy: {hist['acc'][-1]:.1%}")

## Gradient flow through depth

Backpropagation multiplies one Jacobian per layer. In a plain network those products can shrink
the gradient quickly, so early layers learn much more slowly than late ones. A highway layer has a
shortcut: when $T(x)=0$ its Jacobian is the identity, so the gradient can travel through it
unchanged. This is the paper's main argument for why very deep highway networks are easier to
optimize.

Below we measure the gradient that arrives at the input of each block on one batch, before and
after training.

In [ ]:
def layer_input_grad_norms(model, x, y):
    model.train()
    model.zero_grad(set_to_none=True)

    h = model.stem(x)
    h.retain_grad()
    hs = [h]
    for block in model.blocks:
        h = block(h)
        h.retain_grad()
        hs.append(h)

    loss = F.cross_entropy(model.head(h), y)
    loss.backward()

    norms = [max(t.grad.norm().item(), 1e-12) for t in hs[:-1]]
    model.zero_grad(set_to_none=True)
    return norms


set_seed()
fresh_plain = Net(highway=False, width=PLAIN_WIDTH).to(device)
set_seed()
fresh_highway = Net(highway=True, width=HIGHWAY_WIDTH).to(device)

x_grad, y_grad = next(iter(train_loader))
x_grad, y_grad = x_grad[:256].to(device), y_grad[:256].to(device)

fig, (ax_init, ax_trained) = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, title, p_model, h_model in [
    (ax_init, "at initialization", fresh_plain, fresh_highway),
    (ax_trained, "after training", plain, highway),
]:
    ax.semilogy(
        range(1, DEPTH + 1),
        layer_input_grad_norms(h_model, x_grad, y_grad),
        "o-",
        label="highway",
    )
    ax.semilogy(
        range(1, DEPTH + 1),
        layer_input_grad_norms(p_model, x_grad, y_grad),
        "o-",
        label="plain",
    )
    ax.set(xlabel="layer", title=title)
    ax.grid(alpha=0.3, which="both")
    ax.legend()
ax_init.set_ylabel("gradient norm at block input (log scale)")
plt.suptitle("Gradient reaching each layer")
plt.tight_layout()
plt.show()

## Gate bias and gate activity per layer

The paper's Figure 2 inspects two things: the learned transform-gate biases and the mean gate
activity. A negative bias does not necessarily mean that a layer is switched off. It often makes
the gates more selective: some units stay closed while others open for particular inputs.

The bars show the average gate value of each layer over the whole test set.

In [ ]:
def gate_biases(model):
    return [block.gate.bias.mean().item() for block in model.blocks]


@torch.no_grad()
def gate_activity(model, loader):
    model.eval()
    sums = torch.zeros(len(classes), len(model.blocks))
    counts = torch.zeros(len(classes))
    for x, y in loader:
        model(x.to(device))
        batch_means = torch.stack(
            [block.last_gate.mean(dim=1).cpu() for block in model.blocks], dim=1
        )
        for c in range(len(classes)):
            mask = y == c
            sums[c] += batch_means[mask].sum(dim=0)
            counts[c] += mask.sum()
    return sums.sum(dim=0) / counts.sum(), sums / counts[:, None]


overall_activity, class_activity = gate_activity(highway, test_loader)
biases = gate_biases(highway)

fig, (ax_bias, ax_act) = plt.subplots(1, 2, figsize=(11, 3.5))
ax_bias.plot(range(1, DEPTH + 1), biases, "o-", label="learned")
ax_bias.axhline(GATE_BIAS, color="gray", linestyle="--", label="initial")
ax_bias.set(xlabel="layer", ylabel="gate bias", title="Transform-gate bias per layer")
ax_bias.legend()
ax_bias.grid(alpha=0.3)

ax_act.bar(range(1, DEPTH + 1), overall_activity, color="tab:orange")
ax_act.set(
    xlabel="layer",
    ylabel="mean T",
    title="Mean gate activity on the test set",
    ylim=(0, 1),
)
ax_act.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## How the gates change during training

Each row is one highway layer and each column is an epoch. Bright cells mean the layer transforms
its input ($T$ close to 1); dark cells mean it mostly carries the input forward ($T$ close to 0).
The gates start near $\sigma(-2) \approx 0.12$ and move during training.

In [ ]:
gate_history = torch.tensor(highway_hist["gates"]).T  # layers x epochs

plt.figure(figsize=(8, 4))
plt.imshow(gate_history, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="mean transform gate T")
plt.xlabel("epoch")
plt.ylabel("highway layer")
plt.title("Mean gate value of every layer during training")
plt.tight_layout()
plt.show()

## What a single image does to the gates

The paper's Figure 2 inspects the gates of a trained network. Here is the same idea: one test image,
all layers, and all 512 gate values per layer. Many units are strongly closed or strongly open, so
the network uses its gates selectively instead of transforming everything.

In [ ]:
image, label = test_set[0]
with torch.no_grad():
    highway(image.unsqueeze(0).to(device))
gates = []
for block in highway.blocks:
    assert isinstance(block, Highway)
    gates.append(block.last_gate[0].cpu())
gates = torch.stack(gates)

fig, (ax_map, ax_hist) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [3, 1]})
im = ax_map.imshow(gates, aspect="auto", cmap="viridis", vmin=0, vmax=1)
ax_map.set(
    xlabel="hidden unit",
    ylabel="layer",
    title=f"Transform gates for one image ({classes[label]})",
)
fig.colorbar(im, ax=ax_map, label="T")

ax_hist.hist(gates.flatten(), bins=50, range=(0, 1), color="tab:orange")
ax_hist.set(xlabel="T", ylabel="units", title="Gate distribution")
plt.tight_layout()
plt.show()

## Class-specific routing

The gates can route different inputs differently. The paper's Figure 3 compares the mean gate
activity of each class with the overall average. Red cells mean that a class transforms more than
usual in that layer; blue cells mean it carries more than usual.

In [ ]:
diff = (class_activity - overall_activity).numpy()
limit = max(abs(diff).max(), 0.01)

plt.figure(figsize=(9, 4))
plt.imshow(diff, aspect="auto", cmap="coolwarm", vmin=-limit, vmax=limit)
plt.colorbar(label="class mean T - overall mean T")
plt.yticks(range(len(classes)), classes)
plt.xlabel("layer")
plt.title("Which layers each class uses more or less than average")
plt.tight_layout()
plt.show()

## The information highway

A highway layer can pass its input through almost unchanged. The plot below measures how similar
each layer's output is to its input using cosine similarity. Values close to 1 mean the layer
copied its input; values near 0 mean it changed the representation substantially.

In [ ]:
@torch.no_grad()
def layer_similarity(model, x):
    model.eval()
    h = model.stem(x)
    sims = []
    for block in model.blocks:
        h_new = block(h)
        sims.append(F.cosine_similarity(h_new, h, dim=1).mean().item())
        h = h_new
    return sims


plt.figure(figsize=(8, 3.5))
plt.plot(range(1, DEPTH + 1), layer_similarity(highway, probe_x), "o-", label="highway")
plt.plot(range(1, DEPTH + 1), layer_similarity(plain, probe_x), "o-", label="plain")
plt.axhline(0, color="gray", linewidth=1)
plt.ylim(-0.2, 1)
plt.xlabel("layer")
plt.ylabel("cosine similarity(output, input)")
plt.title("How much each layer changes its input")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Lesioning layers

The paper's Figure 4 asks how much each layer actually contributes. We close all transform gates
of one layer at a time, which forces that layer to copy its input, and measure how far test
accuracy drops. A drop near zero means the layer is barely used; a large drop means the layer is
important.

In [ ]:
baseline_acc = accuracy(highway, test_loader)
drops = []
for block in highway.blocks:
    assert isinstance(block, Highway)
    block.gate_off = True
    drops.append(100 * (baseline_acc - accuracy(highway, test_loader)))
    block.gate_off = False

plt.figure(figsize=(8, 3.5))
plt.bar(range(1, DEPTH + 1), drops, color="tab:red")
plt.axhline(0, color="gray", linewidth=1)
plt.xlabel("layer with closed gates")
plt.ylabel("accuracy drop (percentage points)")
plt.title("Contribution of each highway layer")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## Feature space: plain vs. highway

The layer before the classifier is the network's learned representation. A t-SNE plot compresses
that 512-dimensional space to two dimensions. Well-separated clusters mean the representation has
become easier to classify. t-SNE is qualitative, but it is a useful picture of what the extra depth
and gating do to the features.

In [ ]:
@torch.no_grad()
def penultimate_features(model, loader, limit=2000):
    model.eval()
    feats, labels = [], []
    seen = 0
    for x, y in loader:
        h = model.stem(x.to(device))
        for block in model.blocks:
            h = block(h)
        feats.append(h.cpu())
        labels.append(y)
        seen += y.numel()
        if seen >= limit:
            break
    return torch.cat(feats)[:limit].numpy(), torch.cat(labels)[:limit].numpy()


def tsne_2d(feats):
    return TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(feats)


feats_plain, labels_plain = penultimate_features(plain, test_loader)
feats_highway, labels_highway = penultimate_features(highway, test_loader)
emb_plain = tsne_2d(feats_plain)
emb_highway = tsne_2d(feats_highway)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, emb, labels, title in [
    (axes[0], emb_plain, labels_plain, "plain"),
    (axes[1], emb_highway, labels_highway, "highway"),
]:
    for c, name in enumerate(classes):
        mask = labels == c
        ax.scatter(emb[mask, 0], emb[mask, 1], s=5, alpha=0.7, label=name)
    ax.set(title=title, xticks=[], yticks=[])

handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=5, fontsize=7, markerscale=2)
plt.suptitle("t-SNE of penultimate features (2,000 test images)")
plt.tight_layout(rect=(0, 0.08, 1, 1))
plt.show()

## A look at predictions

In [ ]:
images, labels = next(iter(test_loader))
images, labels = images[:8], labels[:8]  # one batch holds 512, show 8
with torch.no_grad():
    probs = highway(images.to(device)).softmax(dim=1).cpu()
labels = labels.tolist()
preds = probs.argmax(dim=1).tolist()
confs = probs.max(dim=1).values.tolist()

fig, axes = plt.subplots(2, 4, figsize=(10, 5.5))
for ax, img, label, pred, conf in zip(axes.flat, images, labels, preds, confs, strict=True):
    rgb = (img * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]).clamp(0, 1)
    ax.imshow(rgb.permute(1, 2, 0))
    ax.set_title(f"true: {classes[label]}\npred: {classes[pred]} ({conf:.0%})", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()